# Module 3 Milestone



## Setup

In [48]:
%pip install transformers sentence-transformers

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


## The scenario and the fixed dataset

In [49]:
messages = [
    "I was charged twice for my subscription this month and I want a refund.",
    "My card was declined at checkout but I was still billed for the upgrade.",
    "The app crashes every time I open the bank reconciliation screen.",
    "After the latest update my reports show the wrong totals, this is broken.",
    "How do I export my invoices to a CSV file?",
    "Where is the setting to add a second user to my account?",
    "Please cancel my subscription, I do not need the service any more.",
    "I want to close my account and stop all future payments.",
]

teams = {
    "Billing":
        "A question or complaint about charges, payments, refunds, invoices, or subscription fees.",

    "Bug report":
        "A report that the software is broken, crashing, showing errors, or giving wrong results.",

    "How-to":
        "A question about how to use a feature or where to find a setting in the product.",

    "Cancellation":
        "A request to cancel the subscription or close the account.",
}

faqs = [
    "To export invoices, open Sales, then Invoices, select the rows, and choose Export to CSV.",
    "To add a user, open Settings, then Manage Users, click Add User, and set their role.",
    "To change your payment method, open Billing, then Payment Methods, and edit the card on file.",
    "To cancel your plan, open Billing, then Subscription, and choose Cancel Subscription.",
    "To reconcile an account, open Accounting, then Reconcile, and pick the account and statement date.",
    "If report totals look wrong, open Reports, then Settings, and refresh the data or clear the cache.",
]

for i, m in enumerate(messages, 1):
    print(i, m)

1 I was charged twice for my subscription this month and I want a refund.
2 My card was declined at checkout but I was still billed for the upgrade.
3 The app crashes every time I open the bank reconciliation screen.
4 After the latest update my reports show the wrong totals, this is broken.
5 How do I export my invoices to a CSV file?
6 Where is the setting to add a second user to my account?
7 Please cancel my subscription, I do not need the service any more.
8 I want to close my account and stop all future payments.


## Part 1. Tokenization audit



In [50]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained("bert-base-uncased")

for i, message in enumerate(messages, 1):
    words = message.split()
    tokens = tok.tokenize(message)

    print(f"Message {i}")
    print("word count:", len(words))
    print("token count:", len(tokens))
    print("tokens:", tokens)
    print()

Message 1
word count: 14
token count: 16
tokens: ['i', 'was', 'charged', 'twice', 'for', 'my', 'subscription', 'this', 'month', 'and', 'i', 'want', 'a', 'ref', '##und', '.']

Message 2
word count: 14
token count: 16
tokens: ['my', 'card', 'was', 'declined', 'at', 'check', '##out', 'but', 'i', 'was', 'still', 'billed', 'for', 'the', 'upgrade', '.']

Message 3
word count: 11
token count: 12
tokens: ['the', 'app', 'crashes', 'every', 'time', 'i', 'open', 'the', 'bank', 'reconciliation', 'screen', '.']

Message 4
word count: 13
token count: 15
tokens: ['after', 'the', 'latest', 'update', 'my', 'reports', 'show', 'the', 'wrong', 'totals', ',', 'this', 'is', 'broken', '.']

Message 5
word count: 10
token count: 14
tokens: ['how', 'do', 'i', 'export', 'my', 'in', '##vo', '##ices', 'to', 'a', 'cs', '##v', 'file', '?']

Message 6
word count: 12
token count: 13
tokens: ['where', 'is', 'the', 'setting', 'to', 'add', 'a', 'second', 'user', 'to', 'my', 'account', '?']

Message 7
word count: 12
toke

### Part 1 takeaway

Some domain-specific words may be split into multiple subword tokens, while common words often remain whole. Token count matters more than ordinary word count because transformer models process text in tokens, and context limits and usage costs are based on tokens rather than words.

## Part 2. Sentiment triage



In [51]:
from transformers import pipeline

sentiment = pipeline("sentiment-analysis")

results = sentiment(messages)

scored_messages = []

for i, (message, result) in enumerate(zip(messages, results), 1):
    label = result["label"]
    score = result["score"]

    high_priority = label == "NEGATIVE" and score > 0.9

    scored_messages.append({
        "number": i,
        "message": message,
        "label": label,
        "score": score,
        "high_priority": high_priority
    })

scored_messages.sort(
    key=lambda x: (
        x["label"] == "NEGATIVE",
        x["score"]
    ),
    reverse=True
)

for item in scored_messages:
    priority = "HIGH" if item["high_priority"] else "normal"

    print(
        f"Message {item['number']}: "
        f"{item['label']} "
        f"{item['score']:.3f} "
        f"| priority: {priority}"
    )

    print(item["message"])
    print()

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use mps:0


Message 4: NEGATIVE 1.000 | priority: HIGH
After the latest update my reports show the wrong totals, this is broken.

Message 8: NEGATIVE 1.000 | priority: HIGH
I want to close my account and stop all future payments.

Message 3: NEGATIVE 1.000 | priority: HIGH
The app crashes every time I open the bank reconciliation screen.

Message 6: NEGATIVE 1.000 | priority: HIGH
Where is the setting to add a second user to my account?

Message 5: NEGATIVE 0.999 | priority: HIGH
How do I export my invoices to a CSV file?

Message 7: NEGATIVE 0.999 | priority: HIGH
Please cancel my subscription, I do not need the service any more.

Message 1: NEGATIVE 0.996 | priority: HIGH
I was charged twice for my subscription this month and I want a refund.

Message 2: NEGATIVE 0.667 | priority: normal
My card was declined at checkout but I was still billed for the upgrade.



### Part 2 limitation

The sentiment model only predicts `POSITIVE` or `NEGATIVE`. It does not have a neutral class, so neutral how-to questions can still be forced into one of the two sentiment labels.

## Part 3. Routing and FAQ retrieval with embeddings



In [52]:
from transformers import AutoTokenizer, AutoModel
import torch
import torch.nn.functional as F

distil_tokenizer = AutoTokenizer.from_pretrained(
    "distilbert-base-uncased"
)

distil_model = AutoModel.from_pretrained(
    "distilbert-base-uncased"
)

In [53]:
def embed_meanpool(text):
    encoded = distil_tokenizer(
        text,
        return_tensors="pt",
        padding=True,
        truncation=True
    )

    with torch.no_grad():
        output = distil_model(**encoded)

    hidden = output.last_hidden_state

    mask = encoded["attention_mask"].unsqueeze(-1)

    masked_hidden = hidden * mask

    summed = masked_hidden.sum(dim=1)

    counts = mask.sum(dim=1)

    mean_vector = summed / counts

    return mean_vector

In [54]:
sentence1 = "How do I export my invoices?"
sentence2 = "How can I download my invoices?"
unrelated = "The application crashes when I reconcile my bank account."

v1 = embed_meanpool(sentence1)
v2 = embed_meanpool(sentence2)
v3 = embed_meanpool(unrelated)

paraphrase_similarity = F.cosine_similarity(
    v1,
    v2
).item()

unrelated_similarity = F.cosine_similarity(
    v1,
    v3
).item()

print(
    f"paraphrase similarity: "
    f"{paraphrase_similarity:.3f}"
)

print(
    f"unrelated similarity: "
    f"{unrelated_similarity:.3f}"
)

paraphrase similarity: 0.944
unrelated similarity: 0.781


The paraphrase pair should have a higher cosine similarity than the unrelated pair because the paraphrase sentences express a more similar meaning.

### Route each message to the nearest team

In [55]:
from sentence_transformers import SentenceTransformer, util

sentence_model = SentenceTransformer("all-MiniLM-L6-v2")

team_names = list(teams.keys())
team_descriptions = list(teams.values())

message_embeddings = sentence_model.encode(
    messages,
    normalize_embeddings=True
)

team_embeddings = sentence_model.encode(
    team_descriptions,
    normalize_embeddings=True
)

similarities = util.cos_sim(
    message_embeddings,
    team_embeddings
)

for i in range(len(messages)):
    best_team_index = similarities[i].argmax().item()
    best_team = team_names[best_team_index]
    best_score = similarities[i][best_team_index].item()

    print(
        f"Message {i + 1} -> "
        f"{best_team} "
        f"(similarity: {best_score:.3f})"
    )

Message 1 -> Billing (similarity: 0.560)
Message 2 -> Billing (similarity: 0.401)
Message 3 -> Bug report (similarity: 0.414)
Message 4 -> Bug report (similarity: 0.351)
Message 5 -> Billing (similarity: 0.236)
Message 6 -> Cancellation (similarity: 0.312)
Message 7 -> Cancellation (similarity: 0.698)
Message 8 -> Cancellation (similarity: 0.603)


### Retrieve the best FAQ for the two how-to questions

In [56]:
faq_embeddings = sentence_model.encode(
    faqs,
    convert_to_tensor=True,
    normalize_embeddings=True
)

# Put FAQ embeddings on the same device as message embeddings
faq_embeddings = faq_embeddings.to(message_embeddings.device)

for message_index in [4, 5]:

    question_embedding = message_embeddings[
        message_index
    ]

    faq_scores = util.cos_sim(
        question_embedding,
        faq_embeddings
    )[0]

    best_faq_index = faq_scores.argmax().item()

    print(f"Message {message_index + 1}:")
    print(messages[message_index])

    print("Best FAQ:")
    print(faqs[best_faq_index])

    print()

Message 5:
How do I export my invoices to a CSV file?
Best FAQ:
To export invoices, open Sales, then Invoices, select the rows, and choose Export to CSV.

Message 6:
Where is the setting to add a second user to my account?
Best FAQ:
To add a user, open Settings, then Manage Users, click Add User, and set their role.



## Part 4. Draft a reply and control the decoding


In [57]:
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    set_seed
)

import torch

gpt_tokenizer = AutoTokenizer.from_pretrained(
    "distilgpt2"
)

gpt_model = AutoModelForCausalLM.from_pretrained(
    "distilgpt2"
)

gpt_tokenizer.pad_token = (
    gpt_tokenizer.eos_token
)

In [58]:
message = messages[4]

prompt = (
    "You are a helpful customer support agent.\n"
    f"Customer: {message}\n"
    "Support reply:"
)

inputs = gpt_tokenizer(
    prompt,
    return_tensors="pt"
)

prompt_tokens = inputs[
    "input_ids"
].shape[1]

context_window = gpt_model.config.n_positions

fraction = prompt_tokens / context_window

print("prompt tokens:", prompt_tokens)
print("context window:", context_window)
print(f"fraction used: {fraction:.4f}")

prompt tokens: 28
context window: 1024
fraction used: 0.0273


### Greedy decoding

In [59]:
set_seed(0)

greedy_output = gpt_model.generate(
    **inputs,
    max_new_tokens=40,
    do_sample=False,
    pad_token_id=gpt_tokenizer.eos_token_id
)

print("GREEDY:")

print(
    gpt_tokenizer.decode(
        greedy_output[0],
        skip_special_tokens=True
    )
)

GREEDY:
You are a helpful customer support agent.
Customer: How do I export my invoices to a CSV file?
Support reply: I export my invoices to a CSV file?
Support reply: I export my invoices to a CSV file?
Support reply: I export my invoices to a CSV file


### High-temperature sampling

In [60]:
set_seed(0)

temperature_output = gpt_model.generate(
    **inputs,
    max_new_tokens=40,
    do_sample=True,
    temperature=1.3,
    top_k=0,
    pad_token_id=gpt_tokenizer.eos_token_id
)

print("HIGH TEMPERATURE:")

print(
    gpt_tokenizer.decode(
        temperature_output[0],
        skip_special_tokens=True
    )
)

HIGH TEMPERATURE:
You are a helpful customer support agent.
Customer: How do I export my invoices to a CSV file?
Support reply: Using button seven.ave3... is "tree" removal the ERSException level and conversion times pre-publically to `promise'. This converts to `WET`ines our Sanskrit search


### Top-p sampling

In [61]:
set_seed(0)

top_p_output = gpt_model.generate(
    **inputs,
    max_new_tokens=40,
    do_sample=True,
    top_p=0.9,
    top_k=0,
    pad_token_id=gpt_tokenizer.eos_token_id
)

print("TOP-P:")

print(
    gpt_tokenizer.decode(
        top_p_output[0],
        skip_special_tokens=True
    )
)

TOP-P:
You are a helpful customer support agent.
Customer: How do I export my invoices to a CSV file?
Support reply: Using this method, you can export my invoices to the CSV file.
Support reply: Using this method, you can export my invoices to the CSV file.
Response: Using


### Part 4 observation

Greedy decoding is deterministic and always chooses the most likely next token, so it can become repetitive. High-temperature sampling introduces more randomness and produces more varied text, but it can wander off-topic. Top-p sampling keeps generation within a smaller set of likely tokens, giving some variety while remaining more controlled.

## Write-up

Tokenization showed that word count and token count are not always the same because the BERT tokenizer can divide less common or domain-specific words into multiple subword pieces. This matters because transformer context limits and processing costs are based on tokens rather than ordinary words. Attention allows each token to use information from other tokens in the sentence, which helps the model interpret meaning and sentiment from context. The same contextual representations help semantically similar support messages and team descriptions appear close together in embedding space, allowing messages to be routed to the appropriate team. Sentence embeddings also allow the system to retrieve FAQ entries whose meaning is similar to a customer's question even when the exact wording is different.